# Mental Health LLM

## 1. Install and Import Libraries

In [1]:
import pandas as pd

from langchain_core.documents import Document
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_community.vectorstores import FAISS

from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.runnables import RunnablePassthrough

import re
from afinn import Afinn

## 2. Establish RAG

In [2]:

csv_file_path = "./train.csv"
df = pd.read_csv(csv_file_path)
print(df[:5])

                                             Context  \
0  I'm going through some things with my feelings...   
1  I'm going through some things with my feelings...   
2  I'm going through some things with my feelings...   
3  I'm going through some things with my feelings...   
4  I'm going through some things with my feelings...   

                                            Response  
0  If everyone thinks you're worthless, then mayb...  
1  Hello, and thank you for your question and see...  
2  First thing I'd suggest is getting the sleep y...  
3  Therapy is essential for those that are feelin...  
4  I first want to let you know that you are not ...  


### 2-1. Concatenate two CSV rows into one row

In [ ]:
description_parts = []

for index, row in df[:5].iterrows():
    for column_name, value in row.items():
        if pd.notna(value):  # Only include non-empty values
            description_parts.append(f"{column_name}: {value}")
print(description_parts)

["Context: I'm going through some things with my feelings and myself. I barely sleep and I do nothing but think about how I'm worthless and how I shouldn't be here.\n   I've never tried or contemplated suicide. I've always wanted to fix my issues, but I never get around to it.\n   How can I change my feeling of being worthless to everyone?", "Response: If everyone thinks you're worthless, then maybe you need to find new people to hang out with.Seriously, the social context in which a person lives is a big influence in self-esteem.Otherwise, you can go round and round trying to understand why you're not worthless, then go back to the same crowd and be knocked down again.There are many inspirational messages you can find in social media. \xa0Maybe read some of the ones which state that no person is worthless, and that everyone has a good purpose to their life.Also, since our culture is so saturated with the belief that if someone doesn't feel good about themselves that this is somehow te

In [ ]:
def make_single_row(row):
    """
    Concatenate two rows into one row
    """
    description_parts = []
    for column_name, value in row.items():
        if pd.notna(value):  # Only include non-empty values
            description_parts.append(f"{column_name}: {value}")
    return ". ".join(description_parts) + "."

In [ ]:
text_doc = []

for index, row in df.iterrows():
    single_row = make_single_row(row)
    doc = Document(page_content=single_row)
    text_doc.append(doc)

print(f"Created {len(text_doc)} Docuemnts objects")

Created 3512 Docuemnts objects


In [ ]:
# Examples of Converted documents

for i in range(min(3, len(text_doc))):
    print(f"Document {i+1}: {text_doc[i].page_content}")

Examples of Converted documents
Document 1: Context: I'm going through some things with my feelings and myself. I barely sleep and I do nothing but think about how I'm worthless and how I shouldn't be here.
   I've never tried or contemplated suicide. I've always wanted to fix my issues, but I never get around to it.
   How can I change my feeling of being worthless to everyone?. Response: If everyone thinks you're worthless, then maybe you need to find new people to hang out with.Seriously, the social context in which a person lives is a big influence in self-esteem.Otherwise, you can go round and round trying to understand why you're not worthless, then go back to the same crowd and be knocked down again.There are many inspirational messages you can find in social media.  Maybe read some of the ones which state that no person is worthless, and that everyone has a good purpose to their life.Also, since our culture is so saturated with the belief that if someone doesn't feel good about

### 2-2. Create Embeddings and Vector Store

In [7]:
embeddings = OllamaEmbeddings(
        model="llama2:latest",
)
print("Embedding system initialized")

Embedding system initialized


### 2-3. Build Vector Search System

In [8]:
text_doc = text_doc[0:100]
vector_store = FAISS.from_documents(text_doc, embeddings)
print(f"Vector store created with {len(text_doc)} documents")

Vector store created with 100 documents


In [9]:
retriever = vector_store.as_retriever()
print(retriever)

tags=['FAISS', 'OllamaEmbeddings'] vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x132df3410> search_kwargs={}


## 3. Connect Model with Prompt

### Prompt Formatting of Llama</br>
<|begin_of_text|><|start_header_id|>system<|end_header_id|></br>
{system_prompt} <|eot_id|></br>
<|start_header_id|>user<|end_header_id|></br>
{user_prompt} <|eot_id|></br>
<|start_header_id|>assistant<|end_header_id|></br>

### 3-1. Load Ollama (I would like to change my own fine tuned model later) 

In [10]:
llm = ChatOllama(
    model="llama2:latest",
    temperature=0,   
)

### 3-2. Structure Prompt

In [11]:
# 원래 코드
# prompt = ChatPromptTemplate.from_messages(
#     [
#         ("system",
#          "You are a helpful assistant. Answer all questions to the best of your ability."
#         ),
#         ("placeholder", "{message}"),
#         ("human", "{input}")  # Setting to save llm messages and User's input separately 
#     ]
# )

# chain = prompt | llm

### 3-3. Connect Model, Prompt and RAG 

In [ ]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser


def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)


prompt = ChatPromptTemplate.from_messages(
    [
        ("system",
         # 여기에 {context}를 추가하여 검색된 정보를 AI에게 줍니다.
         "You are a helpful assistant. Answer questions based on the following context:\n\n{context}"
        ),
        ("placeholder", "{message}"), # 대화 기록(Memory)용
        ("human", "{input}")
    ]
)

chain = (
    {
        "context": retriever | format_docs,  
        "input": RunnablePassthrough(),   
        "message": lambda x: []  
    }
    | prompt 
    | llm
    | StrOutputParser()
)

# response = rag_chain.invoke("input Question")
# print(response)

In [ ]:
result = chain.invoke({"input": "I'm depressed"}, config={"debug": True})
print(result)

In [ ]:
chat1 = prompt.format_messages(input="I'm depressed")
llm.invoke(chat1)

In [ ]:
store = {}

def get_session_history(session_id: str):
    if session_id not in store:
        store[session_id] = ChatMessageHistory() 
    return store[session_id]

In [ ]:
chat_history_for_chain = ChatMessageHistory()

chain_with_message_history = RunnableWithMessageHistory(
    chain,
    get_session_history,           # lambda session_id: chat_history_for_chain,
    input_messages_key="input",    # User's input key
    history_messages_key="message" # llm messages history key
)

print(f"chain_with_message_history: {chain_with_message_history}")
# print(f"chain_with_message_history.invoke: {chain_with_message_history.invoke({"input": "I'm depressed"})}")


In [ ]:
user_input_list = []

while (True):
    user_input = input("Client's Input: ")
    user_input_list.append(user_input)
    
    print(f"Client: {user_input}")
    if user_input.lower() == "exit":
        print("Exiting the chat...")
        break
    response = chain_with_message_history.invoke(
        {"input": user_input},
        {"configurable": {"session_id": "STAR"}},
    )
    print(f"Counselor: {response}")
    print(user_input_list)

In [ ]:
storage = store["STAR"].messages
print(storage)

## 4. Assign Score to Negative Words 

In [ ]:
print(user_input_list)
splt = [w for i in range(2) for w in re.findall(r"[a-zA-Z']+", user_input_list[i].lower())]
print(splt)

In [ ]:
def sentiment_Afinn(words):
    af = Afinn(emoticons=True)
    tuple_list = []
    for i in range(len(words)):
        iscore = af.score(words[i])

        if iscore >= 2:
            iscore = "str_pos"
        elif iscore < 2 and iscore > 0:
            iscore = "pos"
        elif iscore < 0 and iscore > -2:
            iscore = "neg"
        elif iscore <= -2:
            iscore = "str_neg"
        else:
            iscore = "mid"

        tmp = (words[i], iscore)
        tuple_list.append(tmp)
    return tuple_list    

In [ ]:
data = sentiment_Afinn(splt)
print(data)

In [ ]:
neg = [word for word, tag in data if tag == 'neg']
print(f"neg: {neg}")

str_neg = [word for word, tag in data if tag == 'str_neg']
print(f"str_neg: {str_neg}")

## 5. Report to Doctor

In [ ]:
def format_chat_history(messages):
    formatted_text = ""
    for msg in messages:
        role = "Client" if msg.type == "human" else "Counselor"
        formatted_text += f"{role}: {msg.content}\n"
    return formatted_text

full_conversation = format_chat_history(store["STAR"].messages) # If using sessions
print(full_conversation)

In [ ]:
# 1. Define the Prompt
summary_prompt_template = f"""<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are an expert psychological counselor creating a clinical summary report. 
Analyze the following conversation and produce a professional report.<|eot_id|><|start_header_id|>user<|end_header_id|>

Conversation Log:
{full_conversation}

Please provide a "Summary Report" that includes:
1. Client's Main Concerns (Key symptoms or issues mentioned)
2. Emotional State (Observed mood and sentiment)
3. Counselor's Key Advice (Summary of interventions suggested)
4. Recommended Next Steps
5. Count {neg} and {str_neg} in sentence of the Client and write them in a line


Report:<|eot_id|><|start_header_id|>assistant<|end_header_id|>"""

summary_prompt = PromptTemplate(
    template=summary_prompt_template,
    input_variables=["full_conversation"]
)

# Use 'llm' (the base pipeline) here, not 'chat_model', because we are passing a single big block of text, not a chat session.
summary_chain = summary_prompt | llm

In [ ]:
print("\n--- Generating Summary Report... ---\n")

# 1. Get the full history & Summarize it
conversation_text = format_chat_history(chat_history_for_chain.messages)
report = summary_chain.invoke({"conversation": conversation_text})

# print(report)
print(report.content)